# 05 - Classical classifiers on RoBERTa embeddings

A frozen `roberta-base` encoder produces 768-dimensional mean-pooled
embeddings. Logistic regression, SVMs, random forests and LightGBM are then
trained on these features. This isolates the contribution of LoRA fine-tuning
from the contribution of contextual embeddings.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, LinearSVC

from spam_detection.config import (
    CLASSIFIER_TEST_EMBEDDINGS,
    CLASSIFIER_TEST_PATH,
    CLASSIFIER_TRAIN_EMBEDDINGS,
    CLASSIFIER_TRAIN_PATH,
    RESULTS_DIR,
)
from spam_detection.embeddings import get_roberta_embeddings, load_roberta_encoder
from spam_detection.evaluation import binary_metrics, save_table
from spam_detection.utils import set_seed

set_seed(42)

## Load data and embeddings

In [ ]:
train = pd.read_csv(CLASSIFIER_TRAIN_PATH)
test = pd.read_csv(CLASSIFIER_TEST_PATH)

if CLASSIFIER_TRAIN_EMBEDDINGS.exists():
    train_embeddings = np.load(CLASSIFIER_TRAIN_EMBEDDINGS)
else:
    encoder, tokenizer = load_roberta_encoder()
    train_embeddings = get_roberta_embeddings(train["text"].tolist(), model=encoder, tokenizer=tokenizer)
    np.save(CLASSIFIER_TRAIN_EMBEDDINGS, train_embeddings)

if CLASSIFIER_TEST_EMBEDDINGS.exists():
    test_embeddings = np.load(CLASSIFIER_TEST_EMBEDDINGS)
else:
    encoder, tokenizer = load_roberta_encoder()
    test_embeddings = get_roberta_embeddings(test["text"].tolist(), model=encoder, tokenizer=tokenizer)
    np.save(CLASSIFIER_TEST_EMBEDDINGS, test_embeddings)

print(train_embeddings.shape, test_embeddings.shape)

## Train / validation split

In [ ]:
X_train, X_val, y_train, y_val, idx_train, _ = train_test_split(
    train_embeddings,
    train["label"].to_numpy(),
    np.arange(len(train)),
    test_size=0.15,
    random_state=42,
    stratify=train["label"],
)

## Compare classifiers

In [ ]:
models = {
    "RoBERTa + LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RoBERTa + LinearSVM": LinearSVC(class_weight="balanced"),
    "RoBERTa + SVM": SVC(C=1, kernel="rbf", class_weight="balanced"),
    "RoBERTa + RandomForest": RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1),
}

try:
    import lightgbm as lgb

    models["RoBERTa + LightGBM"] = lgb.LGBMClassifier(random_state=42, n_jobs=-1)
except ImportError:
    print("LightGBM is not installed; skipping that baseline.")

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_val)
    scores = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_val)
    metrics = binary_metrics(y_val, predicted, scores)
    rows.append({"Model": name, **{key: round(value, 4) for key, value in metrics.items()}})

results = pd.DataFrame(rows).sort_values("F1", ascending=False)
display(results)

## Final evaluation on the test set

In [ ]:
test_rows = []
for name, model in models.items():
    model.fit(train_embeddings, train["label"])
    predicted = model.predict(test_embeddings)
    scores = model.predict_proba(test_embeddings)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(test_embeddings)
    metrics = binary_metrics(test["label"], predicted, scores)
    test_rows.append({"Model": name, **{key: round(value, 4) for key, value in metrics.items()}})

test_results = pd.DataFrame(test_rows).sort_values("F1", ascending=False)
display(test_results)
save_table(test_results, RESULTS_DIR / "tables" / "baselines_roberta_embeddings.csv")